## Perform Exploration with Data Wrangler

In [2]:
# Azure Open Dataset connection details
blob_account_name = "azureopendatastorage"
blob_container_name = "ojsales-simulatedcontainer"
blob_relative_path = "oj_sales_data"
blob_sas_token = r""  # Blank  since container is Anonymous access

# Set Spark config to access blob storage
wasbs_path = f"wasbs://%s@%s.blob.core.windows.net/%s" % (blob_container_name, blob_account_name, blob_relative_path)
# spark.conf.set("fs.azure.sas.%s.%s.blob.core.windows.net" % (blob_container_name, blob_account_name), blob_sas_token)
spark.conf.set(f"fs.azure.sas.{blob_container_name}.{blob_account_name}.blob.core.windows.net", blob_sas_token)
print("Remote blob path :" + wasbs_path)

# Spark read parquet , note that it won't load any data yet by now
df = spark.read.csv(wasbs_path, header=True)


StatementMeta(, 031bb65b-8324-4d38-8e09-328610541001, 10, Finished, Available, Finished, False)

Remote blob path :wasbs://ojsales-simulatedcontainer@azureopendatastorage.blob.core.windows.net/oj_sales_data


In [3]:
df.head(10)

StatementMeta(, 031bb65b-8324-4d38-8e09-328610541001, 11, Finished, Available, Finished, False)

[Row(WeekStarting='1990-06-14', Store='2384', Brand='minute.maid', Quantity='14579', Advert='1', Price='2.63', Revenue='38342.77'),
 Row(WeekStarting='1990-06-21', Store='2384', Brand='minute.maid', Quantity='15872', Advert='1', Price='2.37', Revenue='37616.64'),
 Row(WeekStarting='1990-06-28', Store='2384', Brand='minute.maid', Quantity='18676', Advert='1', Price='2.46', Revenue='45942.96'),
 Row(WeekStarting='1990-07-05', Store='2384', Brand='minute.maid', Quantity='10986', Advert='1', Price='1.98', Revenue='21752.28'),
 Row(WeekStarting='1990-07-12', Store='2384', Brand='minute.maid', Quantity='14962', Advert='1', Price='2.49', Revenue='37255.380000000005'),
 Row(WeekStarting='1990-07-19', Store='2384', Brand='minute.maid', Quantity='12045', Advert='1', Price='2.64', Revenue='31798.800000000003'),
 Row(WeekStarting='1990-07-26', Store='2384', Brand='minute.maid', Quantity='12588', Advert='1', Price='2.06', Revenue='25931.280000000002'),
 Row(WeekStarting='1990-08-02', Store='2384', 

In [4]:
df.show(10)


StatementMeta(, 031bb65b-8324-4d38-8e09-328610541001, 12, Finished, Available, Finished, False)

+------------+-----+-----------+--------+------+-----+------------------+
|WeekStarting|Store|      Brand|Quantity|Advert|Price|           Revenue|
+------------+-----+-----------+--------+------+-----+------------------+
|  1990-06-14| 2384|minute.maid|   14579|     1| 2.63|          38342.77|
|  1990-06-21| 2384|minute.maid|   15872|     1| 2.37|          37616.64|
|  1990-06-28| 2384|minute.maid|   18676|     1| 2.46|          45942.96|
|  1990-07-05| 2384|minute.maid|   10986|     1| 1.98|          21752.28|
|  1990-07-12| 2384|minute.maid|   14962|     1| 2.49|37255.380000000005|
|  1990-07-19| 2384|minute.maid|   12045|     1| 2.64|31798.800000000003|
|  1990-07-26| 2384|minute.maid|   12588|     1| 2.06|25931.280000000002|
|  1990-08-02| 2384|minute.maid|   14572|     1|  2.4|34972.799999999996|
|  1990-08-09| 2384|minute.maid|   19203|     1| 2.23|          42822.69|
|  1990-08-16| 2384|minute.maid|   16978|     1| 2.14|36332.920000000006|
+------------+-----+-----------+------

In [5]:
import pandas as pd

df = df.toPandas()
df = df.sample(n=500, random_state=1)

df['WeekStarting'] = pd.to_datetime(df['WeekStarting'])
df['Quantity'] = df['Quantity'].astype('int')
df['Advert'] = df['Advert'].astype('int')
df['Price'] = df['Price'].astype('float')
df['Revenue'] = df['Revenue'].astype('float')

df = df.reset_index(drop=True)
df.head(4)

StatementMeta(, 031bb65b-8324-4d38-8e09-328610541001, 13, Finished, Available, Finished, False)

,WeekStarting,Store,Brand,Quantity,Advert,Price,Revenue
0,1991-10-17,947,minute.maid,13306,1,2.42,32200.52
1,1992-03-26,1293,dominicks,18596,1,1.94,36076.24
2,1991-08-15,2278,dominicks,17457,1,2.14,37357.98
3,1992-09-03,2175,tropicana,9652,1,2.07,19979.64


In [6]:
# Code generated by Data Wrangler for pandas DataFrame

def clean_data(df):
    # Replace all instances of "." with " " in column: 'Brand'
    df['Brand'] = df['Brand'].str.replace(".", " ", case=False, regex=False)
    # Capitalize the first character in column: 'Brand'
    df['Brand'] = df['Brand'].str.capitalize()
    return df

df = clean_data(df)

StatementMeta(, 031bb65b-8324-4d38-8e09-328610541001, 14, Finished, Available, Finished, False)

In [7]:
df['Brand'].unique()

StatementMeta(, 031bb65b-8324-4d38-8e09-328610541001, 15, Finished, Available, Finished, False)

array(['Minute maid', 'Dominicks', 'Tropicana'], dtype=object)

## One hot Encoding Transformation

In [8]:
df.head(5)

StatementMeta(, 031bb65b-8324-4d38-8e09-328610541001, 16, Finished, Available, Finished, False)

,WeekStarting,Store,Brand,Quantity,Advert,Price,Revenue
0,1991-10-17,947,Minute maid,13306,1,2.42,32200.52
1,1992-03-26,1293,Dominicks,18596,1,1.94,36076.24
2,1991-08-15,2278,Dominicks,17457,1,2.14,37357.98
3,1992-09-03,2175,Tropicana,9652,1,2.07,19979.64
4,1991-06-27,3372,Dominicks,16587,1,2.26,37486.62


In [9]:
def clean_data(df):
    # Performed 1 aggregation grouped on columns: 'Brand', 'Revenue'
    df = df.groupby(['Brand', 'Revenue']).agg(Revenue_mean=('Revenue', 'mean')).reset_index()
    return df

df = clean_data(df)

StatementMeta(, 031bb65b-8324-4d38-8e09-328610541001, 38, Finished, Available, Finished, False)

In [10]:
print(df)

StatementMeta(, 031bb65b-8324-4d38-8e09-328610541001, 39, Finished, Available, Finished, False)

         Brand   Revenue  Revenue_mean
0    Dominicks  18858.03      18858.03
1    Dominicks  19557.64      19557.64
2    Dominicks  19662.32      19662.32
3    Dominicks  19714.97      19714.97
4    Dominicks  19892.76      19892.76
..         ...       ...           ...
495  Tropicana  49276.96      49276.96
496  Tropicana  49301.20      49301.20
497  Tropicana  49974.10      49974.10
498  Tropicana  50168.30      50168.30
499  Tropicana  51317.82      51317.82

[500 rows x 3 columns]
